# DATA-07: Исследовательский анализ динамического запаздывания отклика качества (Lags & Correlation)

### Контекст задачи
Технологический процесс гидроочистки дизельного топлива (установка 24-2000) и первичной переработки (АВТ-6) обладает существенной инерционностью:
- Гидродинамическое время пребывания сырья в реакторах Р-201/Р-202 составляет 30–50 минут.
- Прохождение через горячий и холодный сепараторы высокого давления (С-201, С-202) и колонну стабилизации К-201 занимает ещё 30–60 минут.
- Суммарное время запаздывания отклика параметров качества (содержание серы, плотность, температура вспышки, фракционный состав) на изменение управляющих параметров КИП (температуры, давления, расходы) составляет **от 30 до 120 минут**.

В данном ноутбуке производится экспериментальный расчет взаимных корреляций телеметрии и показателей качества при сдвигах во времени на **10, 30, 60, 90 и 120 минут** для формирования оптимальных лаговых признаков в моделях Quality Agent (AGENT-02) и Feature Store (DATA-05).

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Настройки графики
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

## 1. Загрузка витрин данных
- `data/processed/telemetry_with_quality.parquet` (DATA-04)
- `data/processed/telemetry_clean.parquet` (DATA-02)

In [ ]:
df_quality = pd.read_parquet('../data/processed/telemetry_with_quality.parquet')
df_telemetry = pd.read_parquet('../data/processed/telemetry_clean.parquet')

if 'date' in df_telemetry.columns:
    df_telemetry['date'] = pd.to_datetime(df_telemetry['date'], utc=True)
    df_telemetry = df_telemetry.sort_values('date').set_index('date')

if 'date' in df_quality.columns:
    df_quality['date'] = pd.to_datetime(df_quality['date'], utc=True)

print(f"Загружено строк телеметрии: {len(df_telemetry):,}")
print(f"Загружено строк качества:    {len(df_quality):,}")

## 2. Формирование сводной таблицы целевых показателей качества

In [ ]:
target_quality = ['Mg.Sulfur', 'Mass.Sulfur', 'D15', '50%.T', '90%.T', '95%.T', 'CFPP', 'FlashPoint']
df_q_valid = df_quality.dropna(subset=['value_quality'])

q_pivot = df_q_valid[df_q_valid['tag'].isin(target_quality)].pivot_table(
    index='date', columns='tag', values='value_quality', aggfunc='mean'
)

q_pivot.describe().T[['count', 'mean', 'std', 'min', '50%', 'max']]

## 3. Расчёт кросс-корреляций со сдвигами на 10, 30, 60, 90, 120 мин
Шаг дискретизации телеметрии составляет 10 минут, поэтому сдвиг на $\tau$ минут соответствует сдвигу на $k = \tau / 10$ точек назад во времени.

In [ ]:
lags_min = [10, 30, 60, 90, 120]
lags_steps = [m // 10 for m in lags_min]

key_telemetry = [
    'T6_hydro', 'T11_hydro', 'T23', 'P8', 'P24', 'F26_hydro', 'W7', 'W10',
    'T55', 'T1', 'T6_avt', 'F7', 'F8', 'F9_avt', 'F30', 'F32', 'F34', 'F56', 'F57', 'F59'
]
key_telemetry = [t for t in key_telemetry if t in df_telemetry.columns]

records = []
for q_tag in q_pivot.columns:
    y = q_pivot[q_tag].dropna()
    if len(y) < 10:
        continue
    for t_tag in key_telemetry:
        x = df_telemetry[t_tag]
        corrs = {}
        for step, lm in zip(lags_steps, lags_min):
            x_shifted = x.shift(step)
            comb = pd.concat([y, x_shifted], axis=1, join='inner').dropna()
            r = float(comb.iloc[:, 0].corr(comb.iloc[:, 1])) if len(comb) >= 10 else 0.0
            corrs[lm] = 0.0 if np.isnan(r) else r
        best_lag = max(corrs.keys(), key=lambda k: abs(corrs[k]))
        records.append({
            'quality_tag': q_tag,
            'telemetry_tag': t_tag,
            'best_lag_min': best_lag,
            'correlation': corrs[best_lag],
            'corr_10m': corrs[10],
            'corr_30m': corrs[30],
            'corr_60m': corrs[60],
            'corr_90m': corrs[90],
            'corr_120m': corrs[120],
        })

df_lags = pd.DataFrame(records)
df_lags.head(10)

## 4. Визуализация: динамика корреляции от времени запаздывания $\text{Corr}(\tau) = f(\tau)$

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# График 1: Сера (Mg.Sulfur) и температуры реактора
ax = axes[0, 0]
for t in ['T6_hydro', 'T11_hydro', 'T23', 'P8']:
    sub = df_lags[(df_lags['quality_tag'] == 'Mg.Sulfur') & (df_lags['telemetry_tag'] == t)]
    if not sub.empty:
        vals = [sub[f'corr_{m}m'].values[0] for m in lags_min]
        ax.plot(lags_min, vals, marker='o', label=t)
ax.set_title('Сера Mg.Sulfur (ПАК): Корреляция с параметрами реактора')
ax.set_xlabel('Лаг запаздывания (мин)')
ax.set_ylabel('Корреляция Пирсона')
ax.legend()
ax.grid(True)

# График 2: Арбитражная сера (Mass.Sulfur ЛИМС)
ax = axes[0, 1]
for t in ['T6_hydro', 'T11_hydro', 'T23', 'F26_hydro']:
    sub = df_lags[(df_lags['quality_tag'] == 'Mass.Sulfur') & (df_lags['telemetry_tag'] == t)]
    if not sub.empty:
        vals = [sub[f'corr_{m}m'].values[0] for m in lags_min]
        ax.plot(lags_min, vals, marker='s', label=t)
ax.set_title('Сера Mass.Sulfur (ЛИМС): Корреляция с гидроочисткой')
ax.set_xlabel('Лаг запаздывания (мин)')
ax.set_ylabel('Корреляция')
ax.legend()
ax.grid(True)

# График 3: Плотность при 15°C (D15)
ax = axes[1, 0]
for t in ['F30', 'F32', 'F34', 'F59']:
    sub = df_lags[(df_lags['quality_tag'] == 'D15') & (df_lags['telemetry_tag'] == t)]
    if not sub.empty:
        vals = [sub[f'corr_{m}m'].values[0] for m in lags_min]
        ax.plot(lags_min, vals, marker='^', label=t)
ax.set_title('Плотность D15: Корреляция с компонентами блендинга')
ax.set_xlabel('Лаг запаздывания (мин)')
ax.set_ylabel('Корреляция')
ax.legend()
ax.grid(True)

# График 4: Фракционный состав 50%.T
ax = axes[1, 1]
for t in ['T55', 'T6_avt', 'F7', 'F30']:
    sub = df_lags[(df_lags['quality_tag'] == '50%.T') & (df_lags['telemetry_tag'] == t)]
    if not sub.empty:
        vals = [sub[f'corr_{m}m'].values[0] for m in lags_min]
        ax.plot(lags_min, vals, marker='d', label=t)
ax.set_title('Температура отгона 50%.T: Корреляция с печью и блендингом')
ax.set_xlabel('Лаг запаздывания (мин)')
ax.set_ylabel('Корреляция')
ax.legend()
ax.grid(True)

plt.tight_layout()
plt.show()

## 5. Итоговый отбор оптимальных лагов (DoD)

In [ ]:
summary = []
for q_tag, grp in df_lags.groupby('quality_tag'):
    best_row = grp.loc[grp['correlation'].abs().idxmax()]
    summary.append({
        'quality_tag': q_tag,
        'primary_telemetry_tag': best_row['telemetry_tag'],
        'recommended_lag_min': int(best_row['best_lag_min']),
        'correlation': round(best_row['correlation'], 4),
    })

df_summary = pd.DataFrame(summary)
df_summary